In [0]:
# ============================================================
# Silver — Source 08: Shopify GraphQL Products
#
# Transformations:
#   - Derive product_sku from handle (sku_00143 → SKU-00143)
#   - Normalise status to uppercase
#   - Normalise product_type to title case
#   - Reject null shopify_id or handle → quarantine
#   - Deduplicate on shopify_id
#
# NOTE: No timestamp column — no watermark dedup possible
#
# Source:  bronze.src_08_shopify.products
# Target:  silver.src_08_shopify.products
# Quarantine: silver.quarantine.src_08_shopify
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from pyspark.sql.window import Window

BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'
TARGET_TABLE = f'{SILVER_CATALOG}.src_08_shopify.products'
QUARANTINE_TABLE = f'{SILVER_CATALOG}.quarantine.src_08_shopify'

VALID_STATUSES = ['ACTIVE', 'DRAFT']

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_08_shopify')
print('Silver Source 08 Shopify — starting...')


In [0]:
# ── LOAD AND CLEAN ────────────────────────────────────────────
bronze = spark.table(f'{BRONZE_CATALOG}.src_08_shopify.products')
total = bronze.count()
print(f'Bronze rows: {total}')

# Step 1: Derive product_sku from handle
# handle format: sku_00143 → SKU-00143
df = bronze.withColumn('product_sku',
    F.upper(F.regexp_replace(F.col('handle'), '_', '-'))
)

# Step 2: Normalise
df = df \
    .withColumn('status',       F.upper(F.trim(F.col('status')))) \
    .withColumn('product_type', F.initcap(F.trim(F.col('product_type'))))

# Step 3: Bad rows
bad = df.filter(
    F.col('shopify_id').isNull() |
    F.col('handle').isNull() |
    F.col('product_sku').isNull() |
    ~F.col('status').isin(VALID_STATUSES)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('products'))

# Step 4: Good rows — dedup on shopify_id
good = df.filter(
    F.col('shopify_id').isNotNull() &
    F.col('handle').isNotNull() &
    F.col('product_sku').isNotNull() &
    F.col('status').isin(VALID_STATUSES)
).dropDuplicates(['shopify_id'])

bad_count = bad.count()
good_count = good.count()
print(f'Shopify products: {total} total → {good_count} clean, {bad_count} quarantined ({bad_count/total*100:.1f}%)')

# Show status distribution
good.groupBy('status').count().show()

# Step 5: Write
if spark.catalog.tableExists(TARGET_TABLE):
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(good.alias('s'), 't.shopify_id = s.shopify_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print('MERGE complete')
else:
    good.write.format('delta').mode('overwrite').saveAsTable(TARGET_TABLE)
    print('Initial load complete')

# Step 6: Quarantine
if bad_count > 0:
    quarantine = bad.select(
        F.lit('src_08_shopify').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in bad.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    )
    quarantine.write.format('delta').mode('append') \
        .option('mergeSchema', 'true').saveAsTable(QUARANTINE_TABLE)
    print(f'✅ {bad_count} rows quarantined')
else:
    print('No quarantine rows')


In [0]:
# ── VERIFY ────────────────────────────────────────────────────
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'silver.src_08_shopify.products: {count} rows')
spark.sql(f"""
    SELECT product_sku, handle, product_type, status
    FROM {TARGET_TABLE}
    LIMIT 5
""").show(truncate=False)
